
# 練習問題: GPUで行列ベクトル積 (行列の map)

## 目標

行列のような **2次元 (大きな) データ** を GPU へ送り, 結果のベクトルを受け取る `map` の使い方を身につける. ベクトルだけを扱った `01_vecadd` と違い, ここでは `n×n` の行列を転送する.

## 課題

行列ベクトル積 `y = A x` を GPU で計算する.

```
y[i] = Σ_j A[i][j] * x[j]
```

C++ では `A` を 1 次元配列とみなし `A[i*n+j]` でアクセスする (Fortran では `A(j,i)` の 2 次元配列).

`gpu_matvec.cpp` (または `gpu_matvec.f90`) の計算本体が抜けている. 初期状態では `y` が番兵 `-1` のままで検算に失敗する. `TODO` の箇所に, **行列ベクトル積を GPU にオフロードして計算する処理** を, 適切な `map` 節とともに書け.

- C++: `#pragma omp target teams distribute parallel for map(to: A[0:n*n], x[0:n]) map(from: y[0:n])` とその直後の二重ループ.
- Fortran: `!$omp target teams distribute parallel do map(to: A, x) map(from: y) private(j, s)` … ループ … `!$omp end target teams distribute parallel do`.

考えどころ:

- 入力 `A` (要素数 `n*n`) と `x` (要素数 `n`) は GPU へ送るだけ (`to`).
- 結果 `y` (要素数 `n`) は GPU から戻すだけ (`from`).
- C++ では配列セクションで転送量を明示する: `A[0:n*n]` は `n²` 要素, `x[0:n]`, `y[0:n]` は `n` 要素. 行列は `n²` に比例して転送量が大きいことを意識せよ.

## コンパイルと実行

```
# C++
nvc++ -mp=gpu gpu_matvec.cpp -o gpu_matvec.exe

# Fortran
nvfortran -mp=gpu gpu_matvec.f90 -o gpu_matvec.exe
```

GPU は計算ノードにのみ搭載されているので `%%bash_submit` でジョブとして投入する.

```
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:10:00

./gpu_matvec.exe 4096
```

## 期待される結果

`A` の全要素を `1`, `x` の全要素を `1` に初期化しているので, 正しく計算できれば `y[i] = n` となり `OK` が表示される.

```
OK: n = 4096, y[0] = 4096 (= n)
```

- `map(to: A ...)` を忘れると GPU 上の `A` が不定になり計算が壊れる.
- `map(from: y)` を忘れるとホスト側の `y` が更新されず番兵 `-1` のままで `NG` になる. 確かめてみよ.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ gpu_matvec.cpp
#include <cstdio>
#include <cstdlib>

int main(int argc, char ** argv) {
  long n = (argc > 1 ? atol(argv[1]) : 4096);
  double * A = (double *)malloc(sizeof(double) * n * n);
  double * x = (double *)malloc(sizeof(double) * n);
  double * y = (double *)malloc(sizeof(double) * n);
  for (long i = 0; i < n; i++) {
    x[i] = 1.0;
    y[i] = -1.0;            /* 番兵: 未計算なら検算に失敗する */
    for (long j = 0; j < n; j++) A[i * n + j] = 1.0;
  }

  /* 行列ベクトル積 y = A x を GPU で計算する.
     A は n*n 要素, x, y は n 要素. A,x は入力 (to:), y は結果 (from:). */
  // TODO: 行列ベクトル積をGPUにオフロードし, A,x は map(to:), 結果 y は map(from:) で受け取れ.

  /* 検算: A[i][j]=1, x[j]=1 なので y[i] = n になるはず */
  long err = 0;
  for (long i = 0; i < n; i++) {
    if (y[i] != (double)n) err++;
  }
  if (err == 0) {
    printf("OK: n = %ld, y[0] = %.0f (= n)\n", n, y[0]);
  } else {
    printf("NG: %ld 要素が不正 (例: y[0] = %.0f, 正解は %ld)\n", err, y[0], n);
  }
  free(A); free(x); free(y);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=gpu gpu_matvec.cpp -o gpu_matvec_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./gpu_matvec_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ gpu_matvec.f90
program gpu_matvec
  character(len=32) :: arg
  integer(8) :: n, i, j, err
  real(8), allocatable :: A(:,:), x(:), y(:)
  real(8) :: s
  n = 4096
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) n
  end if
  allocate(A(n,n), x(n), y(n))
  do i = 1, n
     x(i) = 1.0d0
     y(i) = -1.0d0          ! 番兵: 未計算なら検算に失敗する
     do j = 1, n
        A(j,i) = 1.0d0
     end do
  end do

  ! 行列ベクトル積 y = A x を GPU で計算する.
  ! A は n*n 要素, x, y は n 要素. A,x は入力 (to:), y は結果 (from:).
  ! TODO: 行列ベクトル積をGPUにオフロードし, A,x は map(to:), 結果 y は map(from:) で受け取れ.

  ! 検算: A(j,i)=1, x(j)=1 なので y(i) = n になるはず
  err = 0
  do i = 1, n
     if (y(i) /= dble(n)) err = err + 1
  end do
  if (err == 0) then
     print "(a,i0,a,f0.0,a)", "OK: n = ", n, ", y(1) = ", y(1), " (= n)"
  else
     print "(a,i0,a)", "NG: ", err, " 要素が不正"
  end if
end program gpu_matvec

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=gpu gpu_matvec.f90 -o gpu_matvec_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./gpu_matvec_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:gpu_matvec.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:gpu_matvec.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:gpu_matvec.cpp}